In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df_area = pd.read_csv('datos_occ.csv')
df_area['Timestamp'] = pd.to_datetime(df_area['Timestamp'])

col_intercambio = 'Intercambio neto entre Gerencias (MWh)'

df_area[col_intercambio] = df_area[col_intercambio].astype(str).str.strip()
df_area = df_area[df_area[col_intercambio] != '---'].reset_index(drop=True)
df_area[col_intercambio] = pd.to_numeric(df_area[col_intercambio])

df_area = df_area.sort_values('Timestamp').reset_index(drop=True)

target = 'Estimacion de Demanda por Balance (MWh)'

df_area['hora'] = df_area['Timestamp'].dt.hour
df_area['mes']  = df_area['Timestamp'].dt.month

df_area['demanda_t-1']   = df_area[target].shift(1)
df_area['demanda_t-24']  = df_area[target].shift(24)
df_area['demanda_t-168'] = df_area[target].shift(168)

df_area = df_area.dropna().reset_index(drop=True)

features = ['demanda_t-1', 'demanda_t-24', 'demanda_t-168', 'hora', 'mes']

X = df_area[features]
y = df_area[target]

corte = int(len(df_area) * 0.80)

X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]

def transponer(M):
    return list(map(list, zip(*M)))

def multiplicar_matrices(A, B):
    resultado = [[0 for _ in range(len(B[0]))] for _ in range(len(A))]
    for i in range(len(A)):
        for j in range(len(B[0])):
            for k in range(len(B)):
                resultado[i][j] += A[i][k] * B[k][j]
    return resultado

def matriz_identidad(n):
    I = [[0]*n for _ in range(n)]
    for i in range(n):
        I[i][i] = 1
    return I

def invertir_matriz(M):
    n = len(M)
    I = matriz_identidad(n)
    A = [fila[:] for fila in M]

    for i in range(n):
        pivote = A[i][i]
        for j in range(n):
            A[i][j] /= pivote
            I[i][j] /= pivote
        
        for k in range(n):
            if k != i:
                factor = A[k][i]
                for j in range(n):
                    A[k][j] -= factor * A[i][j]
                    I[k][j] -= factor * I[i][j]
    
    return I

def regresion_lineal_multiple(X, Y):
    X_bias = [[1] + fila for fila in X]
    Y_mat = [[y] for y in Y]

    Xt = transponer(X_bias)
    XtX = multiplicar_matrices(Xt, X_bias)
    XtX_inv = invertir_matriz(XtX)
    XtY = multiplicar_matrices(Xt, Y_mat)

    beta = multiplicar_matrices(XtX_inv, XtY)

    return [b[0] for b in beta]


def predecir_multiple(x, betas):
    resultado = betas[0]
    for i in range(len(x)):
        resultado += betas[i+1] * x[i]
    return resultado

def evaluar_modelo(nombre, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    return {
        'Modelo': nombre,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }

X_train_list = X_train.values.tolist()
y_train_list = y_train.values.tolist()
X_test_list  = X_test.values.tolist()

betas = regresion_lineal_multiple(X_train_list, y_train_list)

print("Coeficientes (betas):")
print(betas)

y_pred = [predecir_multiple(x, betas) for x in X_test_list]

resultado = evaluar_modelo('Regresión Lineal Manual', y_test.values, y_pred)

print("\nResultados:")
print(resultado)

Coeficientes (betas):
[-76.86531892248604, 0.8085860274688645, 0.04172667170472799, 0.16630916106496318, -2.755860417107556, -4.10794093842469]

Resultados:
{'Modelo': 'Regresión Lineal Manual', 'MAE': 165.7236582820487, 'RMSE': np.float64(211.92843379237402), 'R2': 0.9659763084347233}


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df_area = pd.read_csv('datos_occ.csv')
df_area['Timestamp'] = pd.to_datetime(df_area['Timestamp'])

col_intercambio = 'Intercambio neto entre Gerencias (MWh)'

df_area[col_intercambio] = df_area[col_intercambio].astype(str).str.strip()
df_area = df_area[df_area[col_intercambio] != '---'].reset_index(drop=True)
df_area[col_intercambio] = pd.to_numeric(df_area[col_intercambio])

df_area = df_area.sort_values('Timestamp').reset_index(drop=True)

target = 'Estimacion de Demanda por Balance (MWh)'

df_area['hora'] = df_area['Timestamp'].dt.hour
df_area['mes']  = df_area['Timestamp'].dt.month

df_area['demanda_t-1']   = df_area[target].shift(1)
df_area['demanda_t-24']  = df_area[target].shift(24)
df_area['demanda_t-168'] = df_area[target].shift(168)

df_area = df_area.dropna().reset_index(drop=True)

features = ['demanda_t-1', 'demanda_t-24', 'demanda_t-168', 'hora', 'mes']

X = df_area[features]
y = df_area[target]

corte = int(len(df_area) * 0.80)

X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]

def transponer(M):
    return list(map(list, zip(*M)))

def multiplicar_matrices(A, B):
    resultado = [[0 for _ in range(len(B[0]))] for _ in range(len(A))]
    for i in range(len(A)):
        for j in range(len(B[0])):
            for k in range(len(B)):
                resultado[i][j] += A[i][k] * B[k][j]
    return resultado

def matriz_identidad(n):
    I = [[0]*n for _ in range(n)]
    for i in range(n):
        I[i][i] = 1
    return I

def invertir_matriz(M):
    n = len(M)
    I = matriz_identidad(n)
    A = [fila[:] for fila in M]

    for i in range(n):
        pivote = A[i][i]
        for j in range(n):
            A[i][j] /= pivote
            I[i][j] /= pivote
        
        for k in range(n):
            if k != i:
                factor = A[k][i]
                for j in range(n):
                    A[k][j] -= factor * A[i][j]
                    I[k][j] -= factor * I[i][j]
    
    return I
    
def regresion_lineal_multiple(X, Y):
    X_bias = [[1] + fila for fila in X]
    Y_mat = [[y] for y in Y]

    Xt = transponer(X_bias)
    XtX = multiplicar_matrices(Xt, X_bias)
    XtX_inv = invertir_matriz(XtX)
    XtY = multiplicar_matrices(Xt, Y_mat)

    beta = multiplicar_matrices(XtX_inv, XtY)

    return [b[0] for b in beta]


def predecir_multiple(x, betas):
    resultado = betas[0]
    for i in range(len(x)):
        resultado += betas[i+1] * x[i]
    return resultado

def polynomial_features_grado2(X):
    X_poly = []
    
    for fila in X:
        nueva = []
        
        # originales
        nueva.extend(fila)
        
        # cuadrados
        for i in range(len(fila)):
            nueva.append(fila[i] ** 2)
        
        # interacciones
        for i in range(len(fila)):
            for j in range(i+1, len(fila)):
                nueva.append(fila[i] * fila[j])
        
        X_poly.append(nueva)
    
    return X_poly

def evaluar_modelo(nombre, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    return {
        'Modelo': nombre,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }
    
X_train_list = X_train.values.tolist()
y_train_list = y_train.values.tolist()
X_test_list  = X_test.values.tolist()

X_train_poly = polynomial_features_grado2(X_train_list)
X_test_poly  = polynomial_features_grado2(X_test_list)

betas_poly2 = regresion_lineal_multiple(X_train_poly, y_train_list)

print("Número de coeficientes:", len(betas_poly2))

y_pred_poly2 = [predecir_multiple(x, betas_poly2) for x in X_test_poly]

resultado = evaluar_modelo('Polinomial Grado 2 Manual', y_test.values, y_pred_poly2)
print("\nResultados:")
print(resultado)

residuales = y_test.values - y_pred_poly2
hora_test = X_test['hora'].values

print(f"Sesgo: {residuales.mean():.2f}")
print(f"Desviación estándar: {residuales.std():.2f}")

Número de coeficientes: 21

Resultados:
{'Modelo': 'Polinomial Grado 2 Manual', 'MAE': 115.80576563283309, 'RMSE': np.float64(157.94797524875077), 'R2': 0.9811013360952325}
Sesgo: 22.56
Desviación estándar: 156.33


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df_area = pd.read_csv('datos_occ.csv')
df_area['Timestamp'] = pd.to_datetime(df_area['Timestamp'])

col_intercambio = 'Intercambio neto entre Gerencias (MWh)'

df_area[col_intercambio] = df_area[col_intercambio].astype(str).str.strip()
df_area = df_area[df_area[col_intercambio] != '---'].reset_index(drop=True)
df_area[col_intercambio] = pd.to_numeric(df_area[col_intercambio])

df_area = df_area.sort_values('Timestamp').reset_index(drop=True)

target = 'Estimacion de Demanda por Balance (MWh)'

df_area['hora'] = df_area['Timestamp'].dt.hour
df_area['mes']  = df_area['Timestamp'].dt.month

df_area['demanda_t-1']   = df_area[target].shift(1)
df_area['demanda_t-24']  = df_area[target].shift(24)
df_area['demanda_t-168'] = df_area[target].shift(168)

df_area = df_area.dropna().reset_index(drop=True)

features = ['demanda_t-1', 'demanda_t-24', 'demanda_t-168', 'hora', 'mes']

X = df_area[features]
y = df_area[target]

corte = int(len(df_area) * 0.80)

X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]

hora_test = X_test['hora'].values

def evaluar_modelo(nombre, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    return {
        'Modelo': nombre,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }

modelo_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)

resultado = evaluar_modelo('Random Forest', y_test.values, y_pred_rf)

print("\nResultados:")
print(resultado)

residuales_rf = y_test.values - y_pred_rf


print(f"Sesgo: {residuales_rf.mean():.2f}")
print(f"Desviación estándar: {residuales_rf.std():.2f}")


Resultados:
{'Modelo': 'Random Forest', 'MAE': 79.2861684620773, 'RMSE': np.float64(112.6802504835803), 'R2': 0.9903816941492273}
Sesgo: 3.00
Desviación estándar: 112.64
